In [1]:
import torch
import time
import json
import psutil
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import defaultdict

In [2]:
# =============== КОНФИГУРАЦИЯ =================
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
OUTPUT_FILE = "profiling_results.json"
USE_4BIT = False


# ============== ФУНКЦИИ АНАЛИЗА ===============
def get_system_metrics():
    """Собирает данные о железе"""
    gpu_stats = {}
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            gpu_stats[f"GPU_{i}"] = {
                "name": props.name,
                "total_vram_gb": round(props.total_memory / 1024**3, 2)
            }
    return gpu_stats


def analyze_architecture_footprint(model):
    """
    Анализирует, какие слои сколько весят (статический анализ).
    Показывает, куда уходит память.
    """
    print(">>> Анализ архитектуры и распределения весов...")
    param_counts = defaultdict(int)
    param_bytes = defaultdict(int)
    
    total_params = 0
    
    for name, param in model.named_parameters():
        num_params = param.numel()
        total_params += num_params
        
        # Определяем тип блока по имени
        if "embed" in name or "wte" in name:
            block_type = "Embeddings"
        elif "self_attn" in name:
            block_type = "Attention (Q/K/V/O)"
        elif "mlp" in name:
            block_type = "FFN / MLP (Gate/Up/Down)"
        elif "norm" in name:
            block_type = "Normalization"
        elif "lm_head" in name:
            block_type = "LM Head"
        else:
            block_type = "Other"
            
        param_counts[block_type] += num_params
        # считаем размер в ГБ (считаем, что грузим в FP16 = 2 байта, если не квантовано)
        # если модель загружена в 4bit, реальный вес будет меньше, но пропорции те же.
        dtype_size = param.element_size() if hasattr(param, "element_size") else 2
        param_bytes[block_type] += num_params * dtype_size

    # формируем отчет
    breakdown = []
    print(f"{'Block Type':<25} | {'Params (M)':<10} | {'Est. Size (GB)':<15} | {'% of Model'}")
    print("-" * 65)
    
    total_size_gb = sum(param_bytes.values()) / 1024**3
    
    for block, count in param_counts.items():
        size_gb = param_bytes[block] / 1024**3
        percent = (count / total_params) * 100
        print(f"{block:<25} | {count/1e6:<10.1f} | {size_gb:<15.2f} | {percent:.1f}%")
        
        breakdown.append({
            "type": block,
            "params_millions": round(count/1e6, 2),
            "size_gb": round(size_gb, 2),
            "percentage": round(percent, 2)
        })
        
    return breakdown, total_params

In [3]:
# =============== ПРОФИЛИРОВАНИЕ =================
def run_profiling():
    report = {
        "timestamp": time.ctime(),
        "hardware": get_system_metrics(),
        "config": {"use_4bit": USE_4BIT}
    }
    
    print(f"Запуск профилирования на {torch.cuda.device_count()} GPU...")
    
    # 1. ЗАГРУЗКА
    torch.cuda.empty_cache()
    start_load = time.time()
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        
        load_kwargs = {
            "device_map": "auto", 
            "trust_remote_code": True
        }
        
        if USE_4BIT:
            print("Mode: 4-bit Quantization (Colab compatible)")
            load_kwargs["load_in_4bit"] = True
        else:
            print("Mode: FP16 Original (Requires 2xT4 or A100)")
            load_kwargs["torch_dtype"] = torch.float16

        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
        load_time = time.time() - start_load
        print(f"Модель загружена за {load_time:.2f} сек.")
        
    except Exception as e:
        print(f"\nCRITICAL ERROR: Не удалось загрузить модель. Причина: {e}")
        print("Скорее всего, не хватает памяти. Попробуйте USE_4BIT = True")
        return

    # 2. АНАЛИЗ АРХИТЕКТУРЫ
    arch_breakdown, total_params = analyze_architecture_footprint(model)
    report["architecture"] = {
        "total_params_billions": round(total_params/1e9, 2),
        "breakdown": arch_breakdown
    }

    # 3. ЗАМЕР ПАМЯТИ (VRAM)
    vram_usage = []
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.memory_allocated(i) / 1024**3
        print(f"GPU {i} VRAM usage: {mem:.2f} GB")
        vram_usage.append({"gpu_id": i, "used_gb": round(mem, 2)})
    report["memory_static"] = vram_usage

    # 4. ИНФЕРЕНС (СКОРОСТЬ)
    print("\n>>> Тест скорости инференса и вывод ответа...")
    input_text = "Реши уравнение x^2 - 5x + 6 = 0 по шагам."
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    max_gen_tokens = 450

    print("Warming up...")
    _ = model.generate(**inputs, max_new_tokens=10)
    
    # Profiling run
    print(f"Генерация {max_gen_tokens} токенов...")
    torch.cuda.reset_peak_memory_stats()
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=max_gen_tokens, 
            min_new_tokens=max_gen_tokens, # фиксируем для точности замера
            do_sample=False # для детерминированного замера
        )
    end_time = time.time()
    
    # декодирование ответа
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n" + "="*50)
    print("ОТВЕТ МОДЕЛИ (Reasoning + Solution):")
    print("="*50)
    print(generated_text)
    print("="*50 + "\n")
    # ---------------------------------------

    elapsed = end_time - start_time
    
    # расчет метрик
    actual_new_tokens = outputs.shape[1] - inputs.input_ids.shape[1]
    tps = actual_new_tokens / elapsed
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3
    
    print(f"-"*30)
    print(f"Time: {elapsed:.2f} s")
    print(f"Tokens generated: {actual_new_tokens}")
    print(f"Throughput: {tps:.2f} tokens/sec")
    print(f"Peak VRAM during gen: {peak_mem:.2f} GB")
    print(f"-"*30)
    
    report["performance"] = {
        "latency_total_sec": round(elapsed, 2),
        "throughput_tps": round(tps, 2),
        "peak_memory_gb": round(peak_mem, 2),
        "sample_output": generated_text
    }

    # 5. СОХРАНЕНИЕ ОТЧЕТА
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4, ensure_ascii=False)
    print(f"\nОтчет сохранен в файл: {OUTPUT_FILE}")

In [4]:
run_profiling()

Запуск профилирования на 2 GPU...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Mode: FP16 Original (Requires 2xT4 or A100)


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Модель загружена за 333.84 сек.
>>> Анализ архитектуры и распределения весов...
Block Type                | Params (M) | Est. Size (GB)  | % of Model
-----------------------------------------------------------------
Embeddings                | 778.6      | 1.45            | 5.3%
Attention (Q/K/V/O)       | 3020.2     | 5.63            | 20.4%
FFN / MLP (Gate/Up/Down)  | 10192.2    | 18.98           | 69.0%
Normalization             | 0.5        | 0.00            | 0.0%
LM Head                   | 778.6      | 1.45            | 5.3%
GPU 0 VRAM usage: 12.73 GB
GPU 1 VRAM usage: 13.33 GB

>>> Тест скорости инференса и вывод ответа...
Warming up...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Генерация 450 токенов...

ОТВЕТ МОДЕЛИ (Reasoning + Solution):
Реши уравнение x^2 - 5x + 6 = 0 по шагам. Найди корни и проверь их.

</think>

Решение уравнения \( x^2 - 5x + 6 = 0 \):

**Шаг 1:** Напишем уравнение:
\[ x^2 - 5x + 6 = 0 \]

**Шаг 2:** Разложим квадратное уравнение на множители. Для этого найдем два числа, которые в сумме дают \(-5\) и в произведении дают \(6\). Эти числа — \(-2\) и \(-3\).

**Шаг 3:** Запишем уравнение в виде произведения:
\[ (x - 2)(x - 3) = 0 \]

**Шаг 4:** Решим уравнение, приравнивая каждый множитель к нулю:
\[ x - 2 = 0 \quad \text{или} \quad x - 3 = 0 \]
\[ x = 2 \quad \text{или} \quad x = 3 \]

**Корни уравнения:** \( x = 2 \) и \( x = 3 \).

**Проверка:**

Подставим \( x = 2 \) в исходное уравнение:
\[ (2)^2 - 5(2) + 6 = 4 - 10 + 6 = 0 \]

Подставим \( x = 3 \) в исходное уравнение:
\[ (3)^2 - 5(3) + 6 = 9 - 15 + 6 = 0 \]

Оба корня удовлетворяют уравнению. Значит, решение верно. 

**Ответ:** Корни уравнения \( x^2 - 5x + 6 = 0 \) — \( x = 2 \) и